In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as f

In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from ipynb.fs.full.setup import description_from_url, convert_ascii_to_csv

In [4]:
tablename = "department"
url = "https://leetcode.com/problems/reformat-department-table/description/"

In [5]:
print(f"{url}")
print(description_from_url(url))

https://leetcode.com/problems/reformat-department-table/description/
Table: Department

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| revenue     | int     |
| month       | varchar |
+-------------+---------+
In SQL,(id, month) is the primary key of this table.
The table has information about the revenue of each department per month.
The month has values in ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"].

 
Reformat the table such that there is a department id column and a revenue column for each month.
Return the result table in any order.
The result format is in the following example.
 
Example 1:

Input: 
Department table:
+------+---------+-------+
| id   | revenue | month |
+------+---------+-------+
| 1    | 8000    | Jan   |
| 2    | 9000    | Jan   |
| 3    | 10000   | Feb   |
| 1    | 7000    | Feb   |
| 1    | 6000    | Mar   |
+------+---------+-------+
Output: 
+------+-------------+----

In [6]:
convert_ascii_to_csv(f"{tablename}.csv")

Completed writing data to department_converted.csv


In [7]:
spark = SparkSession.builder.appName("Remformat Department Table").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 02:38:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/19 02:38:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
df = spark.read.option("header","true")\
.format("csv")\
.load(f"{tablename}_converted.csv")

In [9]:
df.show()

+---+-------+-----+
| id|revenue|month|
+---+-------+-----+
|  1|   8000|  Jan|
|  2|   9000|  Jan|
|  3|  10000|  Feb|
|  1|   7000|  Feb|
|  1|   6000|  Mar|
+---+-------+-----+



In [10]:
df.createOrReplaceTempView(f"{tablename}")

In [11]:
ws = Window.partitionBy("id")

In [12]:
df.groupBy("id")\
.agg(
    f.sum(f.when(f.col("month")=='Jan', f.col("revenue")).otherwise(f.lit(None))).alias("Jan_Revenue"),
    f.sum(f.when(f.col("month")=='Feb', f.col("revenue")).otherwise(f.lit(None))).alias("Feb_Revenue"),
    f.sum(f.when(f.col("month")=='Mar', f.col("revenue")).otherwise(f.lit(None))).alias("Mar_Revenue"),
    f.sum(f.when(f.col("month")=='Apr', f.col("revenue")).otherwise(f.lit(None))).alias("Apr_Revenue"),
    f.sum(f.when(f.col("month")=='May', f.col("revenue")).otherwise(f.lit(None))).alias("May_Revenue"),
    f.sum(f.when(f.col("month")=='Jun', f.col("revenue")).otherwise(f.lit(None))).alias("Jun_Revenue"),
    f.sum(f.when(f.col("month")=='Jul', f.col("revenue")).otherwise(f.lit(None))).alias("Jul_Revenue"),
    f.sum(f.when(f.col("month")=='Aug', f.col("revenue")).otherwise(f.lit(None))).alias("Aug_Revenue"),
    f.sum(f.when(f.col("month")=='Sep', f.col("revenue")).otherwise(f.lit(None))).alias("Sep_Revenue"),
    f.sum(f.when(f.col("month")=='Oct', f.col("revenue")).otherwise(f.lit(None))).alias("Oct_Revenue"),
    f.sum(f.when(f.col("month")=='Nov', f.col("revenue")).otherwise(f.lit(None))).alias("Nov_Revenue"),
    f.sum(f.when(f.col("month")=='Dec', f.col("revenue")).otherwise(f.lit(None))).alias("Dec_Revenue")
)\
    .orderBy(f.col("id"))\
.toPandas()

,id,Jan_Revenue,Feb_Revenue,Mar_Revenue,Apr_Revenue,May_Revenue,Jun_Revenue,Jul_Revenue,Aug_Revenue,Sep_Revenue,Oct_Revenue,Nov_Revenue,Dec_Revenue
0,1,8000.0,7000.0,6000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,9000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,NaN,10000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
spark.sql(f"""
select id,
sum(case when month='Jan' then revenue else NULL end) as Jan_Revenue,
sum(case when month='Feb' then revenue else NULL end) as Feb_Revenue,
sum(case when month='Mar' then revenue else NULL end) as Mar_Revenue,
sum(case when month='Apr' then revenue else NULL end) as Apr_Revenue,
sum(case when month='May' then revenue else NULL end) as May_Revenue,
sum(case when month='Jun' then revenue else NULL end) as Jun_Revenue,
sum(case when month='Jul' then revenue else NULL end) as Jul_Revenue,
sum(case when month='Aug' then revenue else NULL end) as Aug_Revenue,
sum(case when month='Sep' then revenue else NULL end) as Sep_Revenue,
sum(case when month='Oct' then revenue else NULL end) as Oct_Revenue,
sum(case when month='Nov' then revenue else NULL end) as Nov_Revenue,
sum(case when month='Dec' then revenue else NULL end) as Dec_Revenue
from {tablename}
group by id
order by id asc
""").toPandas()

,id,Jan_Revenue,Feb_Revenue,Mar_Revenue,Apr_Revenue,May_Revenue,Jun_Revenue,Jul_Revenue,Aug_Revenue,Sep_Revenue,Oct_Revenue,Nov_Revenue,Dec_Revenue
0,1,8000.0,7000.0,6000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,9000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,NaN,10000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
